In [1]:
import math
import random
import pandas as pd
from random import randint
random.seed(1)
df = pd.read_excel("../data/points.xlsx")
nodes = {int(row["node"]): (row["x"], row["y"]) for i, row in df.head(20).iterrows()}
# nodes = {int(row["node"]): (row["x"], row["y"]) for i, row in df.iterrows()}


N = len(nodes)
K = 4
Q = 25

demand = {i: random.randint(1,5) for i in nodes.keys() if i!=0}
demand[0] = 0
sum(demand.values())

57

In [2]:
def distance(i, j):
    x1, y1 = nodes[i]
    x2, y2 = nodes[j]

    return int(math.hypot(x1 - x2, y1 - y2)*1000)
    # return math.hypot(x1 - x2, y1 - y2)

In [3]:
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

manager = pywrapcp.RoutingIndexManager(
    N,
    K,
    0
)

routing = pywrapcp.RoutingModel(manager)

In [4]:
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)

    return distance(from_node, to_node)

transit_callback_index = routing.RegisterTransitCallback(distance_callback)

In [5]:
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
# routing.SetFixedCostOfAllVehicles(1000)

In [6]:
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)

    return demand[from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(
    demand_callback
) 

In [7]:
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,
    [Q] * K,
    True,
    "Capacity"
)

True


```
First Solution
    │
    ├── PATH_CHEAPEST_ARC
    ├── SAVINGS
    ├── SWEEP
    └── CHRISTOFIDES

Local Search
    │
    ├── Neighborhood operators
    │       ├── 2-opt
    │       ├── Relocate
    │       ├── Swap
    │       └── ...
    │
    └── Metaheuristic
            ├── Guided Local Search
            ├── Tabu Search
            ├── Simulated Annealing
            └── Greedy Descent

In [8]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()

search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
print("!!")
search_parameters.time_limit.seconds =  5

!!


In [9]:
solution = routing.SolveWithParameters(search_parameters)

In [10]:
if solution:

    for vehicle in range(K):

        index = routing.Start(vehicle)

        route = []
        load = 0
        

        while not routing.IsEnd(index):

            node = manager.IndexToNode(index)

            route.append(node)
            load += demand[node]

            index = solution.Value(
                routing.NextVar(index)
            )

        route.append(manager.IndexToNode(index))

        print(f"Vehicle {vehicle}:")
        print("  Route:", route)
        print("  Load:", load)

Vehicle 0:
  Route: [0, 4, 6, 8, 19, 3, 15, 16, 0]
  Load: 24
Vehicle 1:
  Route: [0, 0]
  Load: 0
Vehicle 2:
  Route: [0, 1, 13, 17, 5, 10, 12, 11, 9, 7, 18, 0]
  Load: 24
Vehicle 3:
  Route: [0, 14, 2, 0]
  Load: 9


In [11]:
objv = solution.ObjectiveValue()/1000
print(objv)
routing.solver().WallTime() / 1000

535.28


5.066

In [12]:
# routes hint for cp_sat
# routes = []
# for k in range(K):
#     index = routing.Start(k)
#     route = []
#     while not routing.IsEnd(index):
#         node = manager.IndexToNode(index)
#         route.append(node)
#         index = solution.Value(routing.NextVar(index))
#     route.append(manager.IndexToNode(index))
#     routes.append(route)
# print(routes)

In [ ]:
# Exact: 
#Objective: 535.280.0
#Wall Time: 102.56657703

#Gap: 0%